# stochastic-rs — CUDA back-end check on a Colab GPU

Runs the CUDA tests of `stochastic-rs-stochastic` on the notebook's GPU, so every CUDA route can be validated without a CUDA machine of your own: the hand-written `cuda` back-end (cuFFT + Philox for fGN, NVRTC kernels for the Euler engine), every process the engine serves — the diffusions, the stochastic-volatility systems, the curve-driven short-rate models, the two jump models, the curve-driven term structures, and the fractional ones, which feed the kernels from the fGN pipeline without a host round trip — including the correlated fractional pair, whose two rows come out of one embedding — and CubeCL's CUDA runtime.

Before running: **Runtime → Change runtime type → T4 GPU** (any NVIDIA GPU works). The first cargo build takes 10–20 minutes on Colab's two cores; the tests themselves take seconds.

Cells: 1 GPU + toolkit check · 2 Rust toolchain · 3 clone · 4 native CUDA tests · 5 every process the engine serves, on the GPU · 6 chunk invariance · 7 (optional) CubeCL CUDA runtime · 8 (optional) Python wheel with `device="cuda"` · 9 (optional) plots from the GPU.

In [ ]:
# 1. The GPU and the CUDA toolkit. The driver's CUDA version (nvidia-smi, top right)
#    should be >= the toolkit's (nvcc): NVRTC emits PTX for the toolkit's version and
#    an older driver cannot JIT it (CUDA_ERROR_UNSUPPORTED_PTX_VERSION). Colab ships
#    them matched; if they differ, see the note at the end.
!nvidia-smi
!nvcc --version | tail -2

In [ ]:
# 2. Rust (stable, minimal profile). PATH is extended for every later cell.
import os
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
os.environ["CARGO_TERM_COLOR"] = "never"
!cargo --version && rustc --version

In [ ]:
# 3. The repository. REF is a branch, tag or commit; main is the default.
REF = "main"
!rm -rf stochastic-rs && git clone --quiet --depth 1 --branch {REF} https://github.com/rust-dd/stochastic-rs.git
%cd stochastic-rs
!git log --oneline -1

In [ ]:
# 4. Native CUDA tests: fGN (cuFFT + Philox; shapes, moments against the CPU
#    path, seed reproducibility regardless of launch history) and the Euler
#    engine (GBM moments, CIR positivity, f32/f64 agreement, chunked batches
#    equal to one launch). The handle is `Cuda`, the feature `cuda`.
#    ~10-20 min to build.
!cargo test -p stochastic-rs-stochastic --features cuda --lib -- cuda 2>&1 | grep -E "^test |^test result|error|panicked"

In [ ]:
# 5. Every process the Euler engine serves, on the GPU. The device-law suite
#    samples each one twice — once on the CPU, once on CUDA — and compares a
#    terminal statistic across thousands of paths, plus whatever boundary the
#    process promises: non-negativity for a square-root diffusion, [0, 1] for
#    the Jacobi and Kimura families, a pinned endpoint for the bridge. It
#    covers the one-component diffusions, the multi-component systems (Heston,
#    SABR, Bergomi, the two-asset Heston, and Heston under both its Euler
#    and quadratic-exponential schemes), the curve-driven short-rate models
#    the jump models and the Levy processes whose increment is a draw rather
#    than a step — a Poisson count, an inverse-Gaussian or gamma draw, a
#    thinned tempered-stable sum — the discrete-time conditional-variance
#    models whose first
#    point is itself a draw, and the fractional processes, whose increments
#    the fGN pipeline writes into a device buffer the kernel steps from, so a
#    batch crosses the bus once. The correlated fractional pair (`Cfgns`,
#    `Cfbms`, `Cfou`) reads two streams out of that one buffer, since both
#    rows share a Hurst exponent and so share an embedding — the case to
#    watch there is the terminal correlation, which a mis-indexed second
#    block loses while leaving both marginals right.
#
#    Two things landed after the last run of this notebook and are worth a
#    look in the output: the dynamic SABR (`MultifactorSabr`) and
#    Heath-Jarrow-Morton (`Hjm`), the two families whose coefficients all
#    arrive as curves rather than parameters — a launch now binds up to eight
#    of them, and the HJM case gives each of its six a distinct time
#    dependence so a slot read from its neighbour moves one row — and
#    `Poisson` in its count mode, whose horizon mode stays on the host and is
#    asserted to. The matrix-valued rate models (`Bgm`, `Adg`, `WuZhangD`)
#    run one launch per row; their cases also check that different rows are
#    uncorrelated on the device, which is what a launch reusing one row's
#    seed for every row would break while leaving every marginal right. The
#    correlated baskets (`MultiGbm`, `Mcgns`, up to four streams) are pinned
#    on the cross-correlation of their terminal log-returns, since that is
#    the one statistic a launch combining the shocks through the wrong
#    Cholesky factor — or none — would move. The one- and two-factor
#    `MultifactorHeston` rides the double-Heston family and is pinned on each
#    factor's correlation with the spot, the one statistic a dropped `rho_k`
#    moves. The regime-switching diffusion (up to four regimes) is pinned on
#    the terminal regime occupancy and the mean switch count, which a chain
#    drawing successors from the wrong row moves while the spot stays
#    plausible. Riemann-Liouville fBm (`RlFBm`) runs its Markov lift in the
#    kernel — the node states of the lift live in per-thread arrays — and is
#    pinned on the spread at the horizon and a quarter of the way in, whose
#    ratio `4^-H` is what tells roughness from Brownian scaling.
!cargo test -p stochastic-rs-stochastic --features cuda --test device_law 2>&1 | grep -E "^test |^test result|error|panicked"


In [ ]:
# 6. Chunk invariance on CUDA. A batch too large for the device budget is
#    launched in several chunks, and the engine promises the result is the
#    same as one launch. For a Gaussian family that holds by construction —
#    the noise is hashed from (path, step, seed) — but a fractional one
#    draws from a pipeline whose counter each launch has to advance itself,
#    and a two-stream launch has to advance it by two rows per path. Neither
#    slip shows up in any statistic of a single batch: the paths still have
#    the right law, they are just repeats or overlaps of each other.
!cargo test -p stochastic-rs-stochastic --features cuda --lib chunks_are_bit_identical 2>&1 | grep -E "^test |^test result|error|panicked"


In [ ]:
# 7. (Optional) CubeCL's CUDA runtime on the same GPU. `family_parity` launches
#    every declared family on both the hand-written NVRTC kernel and the CubeCL
#    one and compares them point for point, which is what keeps the portable
#    route honest as families are added. Adds a few minutes of build time.
RUN_CUBECL = True
if RUN_CUBECL:
    !cargo test -p stochastic-rs-stochastic --features cuda,cubecl-cuda --lib -- cuda cubecl family_parity 2>&1 | grep -E "^test |^test result|error|panicked"


In [ ]:
# 8. (Optional) the Python module with the CUDA back-end: builds the wheel (release,
#    20-30 min on Colab; maturin needs an explicit interpreter here because Colab has
#    no venv) and runs the device-related pytest cases. Set RUN_PYTHON = False to skip
#    the wheel build.
RUN_PYTHON = True
# Add ",cubecl-cuda" to also reach CubeCL's CUDA runtime from Python
# (device="cubecl-cuda"); it costs a few more minutes of build time.
WHEEL_FEATURES = "cuda"
if RUN_PYTHON:
    !pip -q install maturin pytest numpy
    # No virtualenv on Colab, so build the wheel for the notebook's interpreter and pip-install it.
    !maturin build --release --features {WHEEL_FEATURES} --interpreter python3 --out dist 2>&1 | tail -2
    !pip -q install --force-reinstall --no-deps dist/*.whl
    !python -m pytest -q stochastic-rs-py/tests/test_stochastic.py -k "device or probe" 2>&1 | tail -8
    import importlib
    import numpy as np
    sr = importlib.import_module("stochastic_rs")
    names = ["cuda", "cuda:0"] + (["cubecl-cuda"] if "cubecl-cuda" in WHEEL_FEATURES else [])
    for name in names:
        print(f"probe_device({name!r}) ->", sr.probe_device(name))
    # Every device name is one concrete backend: cpu, accelerate, cuda, metal,
    # cubecl-cuda, cubecl-wgpu. There is no name meaning "whatever is compiled".
    try:
        sr.probe_device("gpu")
    except ValueError as e:
        print('probe_device("gpu") ->', e)
    paths = sr.PyGbm(0.05, 0.2, 253, x0=100.0, t=1.0, seed=7, device="cuda").sample_par(20_000)
    print(paths.shape, paths.dtype, "terminal mean / forward =", paths[:, -1].mean() / (100.0 * np.exp(0.05)))

    # A fractional process: the fGN pipeline and the Euler kernel meet in
    # device memory, so this is the path where keeping the increments on the
    # GPU pays. Both runs draw their own stream; the laws must agree.
    import time
    def timed(**kw):
        t = time.perf_counter()
        p = sr.PyFou(0.7, 2.0, 1.0, 0.3, 512, x0=0.0, t=1.0, seed=9, **kw).sample_par(2_000)
        return (time.perf_counter() - t) * 1e3, p
    _ = timed(dtype="f32", device="cuda")
    gpu_ms, gpu = timed(dtype="f32", device="cuda")
    cpu_ms, cpu = timed()
    print(f"fOU 2000x512  cpu {cpu_ms:.1f} ms  cuda {gpu_ms:.1f} ms")
    print(f"terminal mean cpu {cpu[:, -1].mean():.4f}  cuda {gpu[:, -1].mean():.4f}")


In [ ]:
# 9. Plots from the GPU (needs the wheel from cell 8, RUN_PYTHON = True):
#    fBM paths, the t^(2H) variance law GPU vs CPU, the GBM terminal law against
#    the lognormal density, and wall time per call CPU vs CUDA.
if RUN_PYTHON:
    import importlib, time
    import numpy as np
    import matplotlib.pyplot as plt
    sr = importlib.import_module("stochastic_rs")
    print(sr.probe_device("cuda"))
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    # (a) fBM paths for two Hurst exponents, sampled on the GPU (f64).
    ax = axes[0, 0]
    n = 1024
    grid = np.linspace(0.0, 1.0, n)
    for h, color in ((0.3, "tab:red"), (0.8, "tab:blue")):
        paths = sr.PyFbm(h, n, t=1.0, seed=3, device="cuda").sample_par(4)
        for k, path in enumerate(paths):
            ax.plot(grid, path, color=color, lw=0.8, alpha=0.85, label=f"H = {h}" if k == 0 else None)
    ax.set_title("fBM paths, Cuda f64")
    ax.set_xlabel("t")
    ax.legend()

    # (b) Var B_H(t) = t^(2H): GPU dots, CPU crosses, theory lines.
    ax = axes[0, 1]
    m = 4000
    keep = grid > 0.05
    for h, color in ((0.3, "tab:red"), (0.5, "tab:green"), (0.8, "tab:blue")):
        gpu = sr.PyFbm(h, n, t=1.0, seed=11, device="cuda").sample_par(m).var(axis=0)
        cpu = sr.PyFbm(h, n, t=1.0, seed=11, device="cpu").sample_par(m).var(axis=0)
        ax.loglog(grid[keep], gpu[keep], ".", color=color, ms=3, label=f"GPU H={h}")
        ax.loglog(grid[keep], cpu[keep], "x", color=color, ms=3, alpha=0.5, label=f"CPU H={h}")
        ax.loglog(grid[keep], grid[keep] ** (2 * h), "k-", lw=0.8)
    ax.set_title("Var B_H(t) against t^(2H) (black lines), 4000 paths")
    ax.set_xlabel("t")
    ax.legend(fontsize=7, ncol=2)

    # (c) GBM terminal value from the Euler engine against the lognormal density.
    ax = axes[1, 0]
    mu, sigma, s0, horizon = 0.05, 0.2, 100.0, 1.0
    terminal = sr.PyGbm(mu, sigma, 253, x0=s0, t=horizon, seed=7, device="cuda").sample_par(100_000)[:, -1]
    ax.hist(terminal, bins=120, density=True, alpha=0.6, label="Cuda Euler, 100k paths")
    x = np.linspace(terminal.min(), terminal.max(), 400)
    mlog = np.log(s0) + (mu - 0.5 * sigma**2) * horizon
    slog = sigma * np.sqrt(horizon)
    ax.plot(x, np.exp(-(np.log(x) - mlog) ** 2 / (2 * slog**2)) / (x * slog * np.sqrt(2 * np.pi)), "k-", label="lognormal density")
    ax.set_title(f"GBM S_T: sample mean {terminal.mean():.3f}, theory {s0 * np.exp(mu * horizon):.3f}")
    ax.legend()

    # (d) wall time per call, CPU (Colab's two cores) against the T4; warm calls.
    ax = axes[1, 1]
    def timed(fn, warm=1, reps=3):
        for _ in range(warm):
            fn()
        start = time.perf_counter()
        for _ in range(reps):
            fn()
        return (time.perf_counter() - start) / reps
    cases = {
        "fGN n=4096\nm=2000": lambda d: sr.PyFgn(0.7, 4096, t=1.0, seed=1, device=d).sample_par(2000),
        "fGN n=16384\nm=1000": lambda d: sr.PyFgn(0.7, 16384, t=1.0, seed=1, device=d).sample_par(1000),
        "GBM Euler n=253\nm=100k": lambda d: sr.PyGbm(0.05, 0.2, 253, x0=100.0, t=1.0, seed=1, device=d).sample_par(100_000),
    }
    labels, cpu_s, gpu_s = [], [], []
    for name, fn in cases.items():
        labels.append(name)
        cpu_s.append(timed(lambda: fn("cpu")))
        gpu_s.append(timed(lambda: fn("cuda")))
    xs = np.arange(len(labels))
    ax.bar(xs - 0.2, cpu_s, 0.4, label="CPU (Colab, 2 vCPU)")
    ax.bar(xs + 0.2, gpu_s, 0.4, label="Cuda (T4)")
    for i, (c, g) in enumerate(zip(cpu_s, gpu_s)):
        ax.text(i, max(c, g) * 1.15, f"{c / g:.1f}x", ha="center", fontsize=9)
    ax.set_xticks(xs)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_yscale("log")
    ax.set_ylabel("seconds per call (mean of 3, warm)")
    ax.set_title("Wall time per sample_par: CPU vs CUDA")
    ax.legend()
    plt.tight_layout()
    plt.show()


## Reading the result

Every `cuda*` test prints `ok` and the summary line ends in `0 failed`, and cell 5's device-law suite does the same for every process the engine serves. `probe_device("cuda")` reports `backend: 'Cuda'`, and with the CubeCL feature in the wheel `probe_device("cubecl-cuda")` reports `'CubeclCuda'` — the name says which route ran. Two things are worth a look if something is red:

- `CUDA_ERROR_UNSUPPORTED_PTX_VERSION` on module load: the toolkit (nvcc) is newer than the driver. Either pick a runtime whose versions match, or install the toolkit matching `nvidia-smi`'s CUDA version (`apt-get install cuda-toolkit-12-x`) and rerun cell 4.
- `device unavailable: CudaContext: ...`: no GPU in this runtime. Change the runtime type to a GPU and run from cell 1.

The same commands run on any CUDA machine; `STOCHASTIC_RS_DEVICE=n` picks the GPU when there are several, or name it on the handle (`Cuda::new(n)`, `Cubecl::cuda(n)`, `device="cuda:n"`).